# Daily Practice — 2026-11-27 — ML Testing & MLOps: Building a Data Validation Test Suite (Data Contracts)

**Dataset:** [Titanic](https://www.kaggle.com/c/titanic) — 891 real passenger records from the 1912
sinking (class, sex, age, family size, fare, port of embarkation, survival), loaded from the
`mwaskom/seaborn-data` mirror of the classic dataset. It has genuine real-world messiness: ~20% of
`age` values and a couple of `embarked` values are missing, and there is no passenger-ID column.

## Problem statement

In SQA, nothing ships without passing a test suite first. ML pipelines deserve the same discipline —
but most of what breaks a model in production isn't the model code, it's a *data* problem: a column
renamed upstream, a sensor that starts reporting negative values, a new category nobody trained for, a
batch where half the rows are suddenly missing a field. A "data contract" is the test suite for your
input data: a set of checks that run **before** training or scoring, so a bad batch is rejected with a
clear reason instead of silently corrupting a model.

Your job is to build a reusable **data validation test suite** for the Titanic passenger table and prove
it actually catches problems — not just that it runs cleanly on good data.

## What you should produce

1. Five small, independently testable check functions, each returning a list of violation records
   (empty list = no violations):
   - `check_schema(df, expected_kind)` — every expected column is present and its dtype matches the
     expected *kind* (integer / float / string).
   - `check_value_ranges(df, range_specs)` — numeric columns fall within a `(min, max)` range.
   - `check_categorical_domain(df, domain_specs)` — categorical columns only contain allowed values
     (ignoring missing values, which are handled separately).
   - `check_missingness_budget(df, missing_budget)` — the fraction of missing values per column does
     not exceed a per-column budget.
   - `check_duplicates(df, subset=None)` — count exact duplicate rows (optionally on a subset of
     columns).
2. A `run_validation_suite(df)` function that runs all five checks, tags each violation with a severity
   (`BLOCKING` or `WARN` — a given `SEVERITY` dict tells you which check is which), builds a single
   report `DataFrame`, and returns an overall `"PASS"` / `"FAIL"` status — `"FAIL"` only if at least one
   `BLOCKING` violation fired.
3. Proof the suite is not just rubber-stamping everything: run it on the real, clean data (expect a
   `PASS`, possibly with `WARN`-level notes) **and** on a deliberately corrupted copy of a 300-row sample
   batch (expect a `FAIL` that correctly names every injected problem).
4. A short written verdict on the one check this dataset makes genuinely tricky to design: `check_duplicates`.

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option("display.width", 120)

## Load the data

In [ ]:
TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

raw = pd.read_csv(TITANIC_URL)

# Keep only the columns a survival model would actually use as features/target.
# (`deck` is dropped on purpose -- it's ~77% missing and out of scope for this contract.)
COLUMNS = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df = raw[COLUMNS].copy()

print(df.shape)
df.info()
df.head()

## The data contract (given)

These four specs *are* the contract: the shape, ranges, and allowed categories this table must satisfy
before anything downstream (training, scoring) is allowed to touch it. You don't need to change these —
your job is to write the check functions that enforce them.

In [ ]:
# Expected dtype "kind" per column -- checked broadly (integer / float / string) rather than an exact
# dtype string, since pandas can back strings with either plain `object` or a dedicated string dtype.
EXPECTED_KIND = {
    "survived": "integer",
    "pclass": "integer",
    "sex": "string",
    "age": "float",
    "sibsp": "integer",
    "parch": "integer",
    "fare": "float",
    "embarked": "string",
}

# Inclusive (min, max) for numeric columns.
RANGE_SPECS = {
    "age": (0, 100),
    "fare": (0, 600),
    "sibsp": (0, 10),
    "parch": (0, 10),
}

# Allowed categories for categorical columns (missing values are handled by the missingness check,
# not here -- a NaN should never itself be flagged as an "unknown category").
DOMAIN_SPECS = {
    "sex": {"male", "female"},
    "pclass": {1, 2, 3},
    "embarked": {"S", "C", "Q"},
}

# Max tolerated fraction of missing values per column.
MISSING_BUDGET = {
    "age": 0.25,
    "embarked": 0.02,
    "fare": 0.0,
    "sex": 0.0,
    "pclass": 0.0,
    "sibsp": 0.0,
    "parch": 0.0,
}

# Which checks block a release vs. which just warn. `run_validation_suite` should only fail the
# overall status on a BLOCKING violation.
SEVERITY = {
    "schema": "BLOCKING",
    "range": "BLOCKING",
    "domain": "BLOCKING",
    "missingness": "BLOCKING",
    "duplicates": "WARN",
}

## Your task

Fill in the TODOs below. Each check function returns a **list of dicts**, one per violation, with keys
`"check"`, `"column"`, and `"detail"` (a short human-readable string). Returning `[]` means that check
found nothing wrong.

In [ ]:
def check_schema(df, expected_kind):
    """For each (column, kind) in expected_kind: flag a missing column, or a column whose dtype
    doesn't match the expected kind ("integer" / "float" / "string").

    Use pandas type-checking helpers (pd.api.types.is_integer_dtype, is_float_dtype, is_string_dtype)
    rather than comparing dtype strings directly -- pandas can back a string column with more than
    one dtype under the hood, and you don't want a false alarm over that.
    """
    # TODO:
    # 1. For each column not in df.columns -> one violation, detail "missing column".
    # 2. For each column present whose dtype doesn't match the expected kind -> one violation,
    #    detail should mention the kind expected and the dtype actually found.
    raise NotImplementedError

In [ ]:
def check_value_ranges(df, range_specs):
    """For each (column, (lo, hi)) in range_specs: flag rows outside [lo, hi].

    One violation per column (not per row) -- the detail string should say how many rows were
    out of range.
    """
    # TODO: for each column, count rows where value < lo or value > hi. If any, append a violation.
    raise NotImplementedError

In [ ]:
def check_categorical_domain(df, domain_specs):
    """For each (column, allowed_set) in domain_specs: flag any *non-null* value not in allowed_set.

    One violation per column, naming the offending value(s) and how many rows carry them.
    """
    # TODO:
    # 1. Get the non-null unique values actually observed in the column.
    # 2. Subtract the allowed set to find unexpected values.
    # 3. If any, append a violation with a detail naming them and the row count affected.
    raise NotImplementedError

In [ ]:
def check_missingness_budget(df, missing_budget):
    """For each (column, budget) in missing_budget: flag columns whose missing fraction exceeds budget."""
    # TODO: compute df[col].isna().mean() per column and compare against its budget.
    raise NotImplementedError

In [ ]:
def check_duplicates(df, subset=None):
    """Count exact duplicate rows (optionally only comparing `subset` columns).

    Return a single violation if n_dupes > 0, else [].
    """
    # TODO: use df.duplicated(subset=subset) and sum it.
    raise NotImplementedError

In [ ]:
def run_validation_suite(df):
    """Run all five checks, tag every violation with its severity via SEVERITY, and return
    {"status": "PASS" | "FAIL", "report": <DataFrame of violations, with a "severity" column>}.

    Overall status is "FAIL" iff at least one violation has severity "BLOCKING".
    """
    # TODO:
    # 1. Call each check_* function and concatenate their violation lists.
    # 2. Build a DataFrame from the combined list (handle the empty-list case).
    # 3. Add a "severity" column by mapping each row's "check" value through SEVERITY.
    # 4. status = "FAIL" if any row has severity == "BLOCKING", else "PASS".
    raise NotImplementedError

In [ ]:
# TODO: run the suite on the real, clean `df` and print the status and report.
#
# result = run_validation_suite(df)
# print("status:", result["status"])
# print(result["report"])

## Prove the suite actually catches problems

A validation suite that only ever says "PASS" is worthless -- just like a test suite that never fails.
Build a 300-row sample batch (`df.sample(n=300, random_state=RANDOM_STATE)`) and corrupt a copy of it
exactly like this, using `rng` (the `np.random.default_rng(RANDOM_STATE)` from the Setup cell) so your
results are reproducible:

1. Set `fare` to `-50.0` on 5 rows chosen with `rng.choice(bad.index, size=5, replace=False)`.
2. Set `sex` to `"unknown"` on 4 *different* rows (same `rng.choice` pattern, `size=4`).
3. Set `pclass` to `9` on 3 more rows (`size=3`).
4. Set `age` to `NaN` on 120 more rows (`size=120`) -- on top of the real missingness already there.

Run `run_validation_suite` on the corrupted batch and confirm it reports `"FAIL"` and names all four
injected problems (plus whatever the duplicates check says).

In [ ]:
# TODO: build `bad` (the corrupted 300-row batch) following the recipe above, then run the suite
# on it and print the status and report.
#
# bad = df.sample(n=300, random_state=RANDOM_STATE).copy()
# idx = bad.index
# ...

### Write-up

TODO: in 3-5 sentences, answer: why is `check_duplicates` the one check in this suite you *can't* fully
trust on this dataset? (Hint: look at how many "duplicate" rows the real, uncorrupted data already has,
and think about what a passenger table is missing that would make a duplicate row actually mean
something.) Given that, do you agree `duplicates` should be `WARN` rather than `BLOCKING` severity? What
would you need to add to this dataset to make a `BLOCKING` duplicate check trustworthy?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Schema check
def check_schema(df, expected_kind):
    def matches(series, kind):
        if kind == "integer":
            return pd.api.types.is_integer_dtype(series)
        if kind == "float":
            return pd.api.types.is_float_dtype(series)
        if kind == "string":
            return pd.api.types.is_string_dtype(series) or series.dtype == object
        return False

    violations = []
    for col, kind in expected_kind.items():
        if col not in df.columns:
            violations.append({"check": "schema", "column": col, "detail": "missing column"})
        elif not matches(df[col], kind):
            violations.append({"check": "schema", "column": col,
                                "detail": f"expected {kind}-like dtype, got {df[col].dtype}"})
    return violations

# 2. Range check
def check_value_ranges(df, range_specs):
    violations = []
    for col, (lo, hi) in range_specs.items():
        bad_rows = df[(df[col] < lo) | (df[col] > hi)]
        if len(bad_rows):
            violations.append({"check": "range", "column": col,
                                "detail": f"{len(bad_rows)} rows outside [{lo}, {hi}]"})
    return violations

# 3. Categorical domain check
def check_categorical_domain(df, domain_specs):
    violations = []
    for col, allowed in domain_specs.items():
        observed = set(df[col].dropna().unique())
        bad_values = observed - allowed
        if bad_values:
            n_bad = df[col].isin(bad_values).sum()
            violations.append({"check": "domain", "column": col,
                                "detail": f"{n_bad} rows with values outside {sorted(allowed)}: {sorted(bad_values)}"})
    return violations

# 4. Missingness budget check
def check_missingness_budget(df, missing_budget):
    violations = []
    for col, budget in missing_budget.items():
        frac = df[col].isna().mean()
        if frac > budget:
            violations.append({"check": "missingness", "column": col,
                                "detail": f"{frac:.1%} missing exceeds budget {budget:.1%}"})
    return violations

# 5. Duplicates check
def check_duplicates(df, subset=None):
    n_dupes = df.duplicated(subset=subset).sum()
    label = "full-row" if subset is None else f"subset={subset}"
    if n_dupes:
        return [{"check": "duplicates", "column": label, "detail": f"{n_dupes} duplicate rows"}]
    return []

# 6. The suite itself
def run_validation_suite(df):
    violations = []
    violations += check_schema(df, EXPECTED_KIND)
    violations += check_value_ranges(df, RANGE_SPECS)
    violations += check_categorical_domain(df, DOMAIN_SPECS)
    violations += check_missingness_budget(df, MISSING_BUDGET)
    violations += check_duplicates(df)
    report = pd.DataFrame(violations)
    if not report.empty:
        report["severity"] = report["check"].map(SEVERITY)
    status = "FAIL" if not report.empty and (report["severity"] == "BLOCKING").any() else "PASS"
    return {"status": status, "report": report}

# 7. Run on real, clean data
result = run_validation_suite(df)
print("status:", result["status"])
print(result["report"])

# 8. Build and run on the corrupted batch
bad = df.sample(n=300, random_state=RANDOM_STATE).copy()
idx = bad.index
bad.loc[rng.choice(idx, size=5, replace=False), "fare"] = -50.0
bad.loc[rng.choice(idx, size=4, replace=False), "sex"] = "unknown"
bad.loc[rng.choice(idx, size=3, replace=False), "pclass"] = 9
bad.loc[rng.choice(idx, size=120, replace=False), "age"] = np.nan

result_bad = run_validation_suite(bad)
print("status:", result_bad["status"])
print(result_bad["report"])
```

**Typical output**

```
status: PASS
     check   column             detail severity
duplicates full-row 111 duplicate rows     WARN

status: FAIL
      check   column                                                     detail severity
      range     fare                                    5 rows outside [0, 600] BLOCKING
     domain      sex 4 rows with values outside ['female', 'male']: ['unknown'] BLOCKING
     domain   pclass        3 rows with values outside [1, 2, 3]: [9]            BLOCKING
missingness      age                         53.0% missing exceeds budget 25.0% BLOCKING
 duplicates full-row                                          36 duplicate rows     WARN
```

**Explanation**

The real, uncorrupted data passes cleanly — the only thing the suite has to say is a `WARN` about 111
full-row duplicates out of 891 rows (12.5%). That number is misleading, not alarming: the Titanic table
has no passenger-ID column, so "duplicate" here just means two different real passengers who happened to
share `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, and `embarked` exactly — entirely plausible for,
say, two 24-year-old third-class men traveling alone who paid the same low fare and boarded at
Southampton. A `check_duplicates` without a primary key is really a coincidence detector, not a data-
integrity detector, which is exactly why it's marked `WARN` rather than `BLOCKING`: blocking a release
over it would reject perfectly good batches purely because the schema is missing a unique row
identifier. To make it trustworthy at `BLOCKING` severity you'd need the source system to emit a genuine
passenger/ticket ID (or at minimum a `ticket` number, which the full dataset has but this trimmed
feature table doesn't) and check duplicates on *that* subset instead of the full row.

On the corrupted batch, the suite correctly fires on every injected problem and nothing else: the 5
negative fares, the 4 `"unknown"` sex values, the 3 `pclass=9` rows, and the `age` column's missingness
jumping from ~20% to 53% (well past its 25% budget) each produce exactly one `BLOCKING` row. The
duplicates check still only warns — and the duplicate count actually *drops* (111 → 36) simply because
the batch is smaller, which is itself a small lesson: raw duplicate counts don't mean much without
normalizing for batch size, another reason this check belongs at `WARN`, not `BLOCKING`.

</details>